# Analisis de Resultados - Buscador Semantico Simple

**Proyecto 2 — Algebra Lineal para la Computacion**  
EID 2026

Experimentos con TF (frecuencia de terminos) y TF-IDF para recuperacion de documentos.

In [ ]:
import sys
import numpy as np
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.core.preprocessing import preprocess
from src.core.vectorizer import build_vocabulary, build_tf_matrix
from src.core.similarity import cosine_similarity, cosine_similarities
from src.core.search import search

# funciones de visualizacion inline (independientes de src/heatmap y src/term_frequency)
def plot_similarity_heatmap(sim_matrix, doc_names, save_path=None):
    import matplotlib.pyplot as plt
    import seaborn as sns
    plt.figure(figsize=(10, 8))
    sns.heatmap(sim_matrix, xticklabels=doc_names, yticklabels=doc_names,
                annot=True, fmt=".2f", cmap="RdBu_r", square=True)
    plt.title("Matriz de Similitudes entre Documentos")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    return plt.gcf()

def plot_top_terms(matrix, vocab, top_n=20, save_path=None):
    import matplotlib.pyplot as plt
    import numpy as np
    freqs = np.array(matrix).sum(axis=0)
    idx = np.argsort(freqs)[::-1][:top_n]
    words = [vocab[i] for i in idx]
    vals = freqs[idx]
    plt.figure(figsize=(10, 6))
    plt.barh(range(len(words)), vals, color="steelblue")
    plt.yticks(range(len(words)), words)
    plt.xlabel("Frecuencia")
    plt.title(f"Top-{top_n} Terminos mas Frecuentes")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    return plt.gcf()

## 1. Carga y preprocesamiento del corpus

In [ ]:
def load_corpus(corpus_dir="../data/corpus/"):
    paths = sorted(Path(corpus_dir).glob("*.txt"))
    names, documents = [], []
    for path in paths:
        with open(path, "r", encoding="utf-8") as f:
            documents.append(f.read())
        names.append(path.stem)
    return names, documents

doc_names, raw_docs = load_corpus()
corpus_tokens = [preprocess(doc) for doc in raw_docs]
vocabulary = build_vocabulary(corpus_tokens)
tf_matrix = build_tf_matrix(corpus_tokens, vocabulary)

print(f"Documentos cargados: {len(doc_names)}")
print(f"Vocabulario: {len(vocabulary)} palabras unicas")

## 2. Consultas experimentales (Top-3)

In [ ]:
queries = [
    "inteligencia artificial y aprendizaje automatico",
    "calentamiento global y energias renovables",
    "guerra mundial historia antigua civilizaciones",
    "matrices vectores y algebra lineal",
    "cambio climatico deforestacion y perdida de especies",
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"Consulta: {q}")
    print('='*60)
    results = search(corpus_tokens, preprocess(q), method="tf", top_k=3)
    for idx, score in results:
        print(f"  {doc_names[idx]:30s}  {score:.4f}")

## 3. Matriz de similitud entre documentos (Heatmap)

In [ ]:
n = len(corpus_tokens)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_similarity(tf_matrix[i], tf_matrix[j])

fig = plot_similarity_heatmap(sim_matrix, doc_names, save_path="../docs/figuras/heatmap_tf.png")
print("Heatmap guardado en docs/figuras/heatmap_tf.png")

## 4. Frecuencia de terminos (Top-20)

In [ ]:
fig = plot_top_terms(tf_matrix, vocabulary, top_n=20, save_path="../docs/figuras/top_term_freq.png")
print("Grafico de frecuencias guardado en docs/figuras/top_term_freq.png")

## 5. Comparacion TF vs TF-IDF

TF-IDF pondera cada termino segun su frecuencia inversa en el corpus, reduciendo el peso de palabras muy comunes.
Esta seccion se ejecutara cuando la funcion `build_tfidf_matrix` este disponible en `vectorizer.py`.

In [ ]:
try:
    from src.core.vectorizer import build_tfidf_matrix
    tfidf_matrix = build_tfidf_matrix(corpus_tokens, vocabulary)
    print(f"Matriz TF-IDF: {tfidf_matrix.shape}")

    print("\nComparacion TF vs TF-IDF para la consulta:\n")
    for q in queries[:2]:
        q_tokens = preprocess(q)
        tf_results = search(corpus_tokens, q_tokens, method="tf", top_k=3)
        print(f"\n--- {q} ---")
        print("  TF:")
        for idx, s in tf_results:
            print(f"    {doc_names[idx]:30s}  {s:.4f}")

        # TF-IDF manual
        from src.core.vectorizer import build_vocabulary, build_tf_matrix
        from src.core.similarity import cosine_similarity
        vocab_tfidf = build_vocabulary(corpus_tokens)
        q_vec_tfidf = build_tfidf_matrix([q_tokens], vocab_tfidf)[0]
        scores = [(i, cosine_similarity(q_vec_tfidf, tfidf_matrix[i])) for i in range(len(corpus_tokens))]
        scores.sort(key=lambda x: x[1], reverse=True)
        print("  TF-IDF:")
        for idx, s in scores[:3]:
            print(f"    {doc_names[idx]:30s}  {s:.4f}")

    sim_matrix_tfidf = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            sim_matrix_tfidf[i, j] = cosine_similarity(tfidf_matrix[i], tfidf_matrix[j])
    fig = plot_similarity_heatmap(sim_matrix_tfidf, doc_names, save_path="../docs/figuras/heatmap_tfidf.png")
    print("\nHeatmap TF-IDF guardado en docs/figuras/heatmap_tfidf.png")

except ImportError:
    print("[TF-IDF no disponible aun — se ejecutara cuando se implemente build_tfidf_matrix]")

## 6. Discusion

### Observaciones

- La representacion Bag-of-Words pierde el orden de las palabras pero captura la tematica general.
- Documentos del mismo tema (ej. tecnologia_01 y tecnologia_04) muestran mayor similitud.
- TF-IDF mejora los resultados al penalizar terminos muy frecuentes en todo el corpus.
- El coseno es robusto ante diferencias en la longitud de los documentos (normaliza por norma).

### Limitaciones

- No entiende sinonimos ni contexto (vectores ortogonales para palabras distintas aunque signifiquen lo mismo).
- Sensible al preprocesamiento: palabras mal tokenizadas o acentos no eliminados afectan el resultado.
- Vocabulario pequeno con solo 16 documentos; con mas datos la escasez (sparsity) seria mayor.